# Phase 4a: Random Forest Model (Week 10)

Train and evaluate a Random Forest model on combined user-destination features.

**Input:** Phases 1-3 cached data (user/destination profiles + weather + embeddings)

**Output:** Trained model + evaluation metrics (Recall@10, NDCG@10, MRR)

In [ ]:
import sys
sys.path.insert(0, '../../')

import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from config import *
from src.models.random_forest import RandomForestRecommender
from src.evaluation.harness import EvaluationHarness

print("="*70)
print("PHASE 4a: Random Forest Model")
print("="*70)

## Step 1: Load Phase 1-3 Cached Data

In [ ]:
print("\n[1/6] Loading Phase 1 data...")

# Load training and test sets
train_reviews = pd.read_parquet(CACHE_DIR / 'train_reviews.parquet')
test_reviews = pd.read_parquet(CACHE_DIR / 'test_reviews.parquet')
businesses = pd.read_parquet(CACHE_DIR / 'businesses_major_cities.parquet')

print(f"  ✓ Training reviews: {len(train_reviews):,}")
print(f"  ✓ Test reviews: {len(test_reviews):,}")
print(f"  ✓ Businesses: {len(businesses):,}")
print(f"  ✓ Cities: {businesses['city'].nunique()}")

## Step 2: Load Weather Features (Phase 2)

In [ ]:
print("\n[2/6] Loading Phase 2 weather features...")

weather_features = pd.read_parquet(CACHE_DIR / 'weather_features_for_models.parquet')
print(f"  ✓ Weather features: {weather_features.shape}")
print(f"  ✓ Features per city: {weather_features.shape[1] - 3}")

## Step 3: Load Embeddings (Phase 3)

In [ ]:
print("\n[3/6] Loading Phase 3 embeddings...")

dest_embeddings = joblib.load(CACHE_DIR / 'destination_embeddings.joblib')
user_embeddings = joblib.load(CACHE_DIR / 'user_embeddings.joblib')

print(f"  ✓ Destination embeddings: {len(dest_embeddings)} cities")
print(f"  ✓ User embeddings: {len(user_embeddings)} users")
print(f"  ✓ Embedding dimension: 384")

## Step 4: Build User and Destination Feature DataFrames

In [ ]:
print("\n[4/6] Building feature dataframes...")

# Create user features from training data
user_features_list = []
for user_id in train_reviews['user_id'].unique():
    user_reviews = train_reviews[train_reviews['user_id'] == user_id]
    
    user_feat = {
        'user_id': user_id,
        'avg_rating': user_reviews['rating'].mean(),
        'rating_std': user_reviews['rating'].std(),
        'num_reviews': len(user_reviews),
        'num_cities': user_reviews['city'].nunique()
    }
    user_features_list.append(user_feat)

user_features_df = pd.DataFrame(user_features_list)
print(f"  ✓ User features: {user_features_df.shape}")

# Create destination features
dest_features_list = []
for city in businesses['city'].unique():
    city_businesses = businesses[businesses['city'] == city]
    city_reviews = train_reviews[train_reviews['city'] == city]
    
    dest_feat = {
        'city': city,
        'avg_rating': city_reviews['rating'].mean() if len(city_reviews) > 0 else 0,
        'num_reviews': len(city_reviews),
        'num_businesses': len(city_businesses),
    }
    dest_features_list.append(dest_feat)

dest_features_df = pd.DataFrame(dest_features_list)

# Merge with weather features
dest_features_df = dest_features_df.merge(
    weather_features,
    left_on='city',
    right_on='city',
    how='left'
)

print(f"  ✓ Destination features: {dest_features_df.shape}")

## Step 5: Train Random Forest Model

In [ ]:
print("\n[5/6] Training Random Forest model...")

# Initialize recommender
rf_recommender = RandomForestRecommender(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    random_state=RF_RANDOM_STATE
)

# Build feature matrix for training
X_train, y_train = rf_recommender.build_feature_matrix(
    train_reviews,
    user_features_df,
    dest_features_df,
    dest_embeddings,
    user_embeddings
)

print(f"  ✓ Training set: {X_train.shape}")
print(f"  ✓ Features: {X_train.shape[1]}")
print(f"  ✓ Samples: {X_train.shape[0]}")

# Train model
rf_recommender.train(X_train, y_train)
print(f"  ✓ Model trained with {RF_N_ESTIMATORS} trees, max_depth={RF_MAX_DEPTH}")

## Step 6: Evaluate on Test Set

In [ ]:
print("\n[6/6] Evaluating on test set...")

# Build feature matrix for test
X_test, y_test = rf_recommender.build_feature_matrix(
    test_reviews,
    user_features_df,
    dest_features_df,
    dest_embeddings,
    user_embeddings
)

print(f"  ✓ Test set: {X_test.shape}")

# Predict
y_pred = rf_recommender.predict(X_test)

# Compute evaluation metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\n{'='*70}")
print("RANDOM FOREST RESULTS")
print(f"{'='*70}")
print(f"Mean Squared Error (MSE):  {mse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R² Score:                  {r2:.4f}")
print(f"{'='*70}")

## Feature Importance Analysis

In [ ]:
print("\nTop 20 Most Important Features:")
print("-" * 50)

top_features = rf_recommender.get_feature_importance(top_n=20)
if top_features is not None:
    for idx, row in top_features.iterrows():
        print(f"{row['feature']:30s} {row['importance']:.4f}")
else:
    print("Feature importance not available")

## Save Model for Phase 5

In [ ]:
print("\nSaving trained model...")

model_path = OUTPUT_DIR / 'random_forest_model.joblib'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save model and preprocessors
joblib.dump(rf_recommender, model_path)
print(f"  ✓ Model saved to {model_path}")

# Save results
results = {
    'mse': float(mse),
    'mae': float(mae),
    'r2': float(r2),
    'n_estimators': RF_N_ESTIMATORS,
    'max_depth': RF_MAX_DEPTH,
    'n_features': X_train.shape[1],
    'n_train_samples': X_train.shape[0],
    'n_test_samples': X_test.shape[0]
}

results_path = REPORT_DIR / 'phase4a_random_forest_results.txt'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

with open(results_path, 'w') as f:
    f.write("PHASE 4a: Random Forest Results\n")
    f.write("="*50 + "\n")
    for k, v in results.items():
        f.write(f"{k:20s}: {v}\n")

print(f"  ✓ Results saved to {results_path}")

## Summary

✅ **Phase 4a Complete: Random Forest Model**

**Model Configuration:**
- Estimators: 100
- Max depth: 15
- Features: ~800 dimensions (user + destination + embeddings + weather)

**Performance:**
- MSE: [calculated above]
- MAE: [calculated above]
- R²: [calculated above]

**Next Steps:**
- Phase 4b: Train Two-Tower Neural Network
- Phase 5: Run ablation experiments
- Phase 6: Final analysis